# Detecting Appreciation-Led Pricing in Texas Housing Markets Using User-Cost Theory
**By Chinedu Okafor**

**Last Modified:** 5/13/2026

---
## Executive Summary
This analysis evaluates whether Texas housing markets were priced consistently with rent-implied fundamentals between 2010–2019 using a user-cost valuation framework.

### Core Findings
- 60% of valid metro-year observations fell into appreciation-led regimes
- Dallas exhibited persistent divergence from rent-based valuation
- Houston remained comparatively fundamentals-aligned
- Rent-based equilibrium explained only a limited subset of pricing behavior

### Analytical Methods
- Multi-source data integration (Zillow + FRED)
- Valuation modeling
- Cross-market comparative analysis
- Sensitivity testing

### Key Takeaway
Texas housing price dynamics during the 2010s appear increasingly difficult to explain through rent-based fundamentals alone, suggesting expectations of continued appreciation played a meaningful role in market pricing.

---
## Overview
This notebook investigates whether Texas housing markets (Dallas, Houston, Austin, and San Antonio) were priced consistently with rent-implied valuation benchmarks, and whether buyers were implicitly relying on future appreciation to justify purchase decisions.

### The Central Question
> *For a given city and year: what annual home price growth did the market have to
> assume for buying to make more sense than renting — and was the market operating
> on rental fundamentals at all, or driven purely by appreciation expectations?*

We answer this using a rent-vs-own equilibrium formula, then compare the market's implied expectation against what actually happened over the next 5 years.

---

### Data Sources

| Dataset | Source | What it contains |
|---|---|---|
| **ZHVI** | Zillow | Median home prices by metro area, monthly |
| **ZORI** | Zillow | Median monthly rents by metro area |
| **FRED** | Federal Reserve | Weekly 30-year fixed mortgage rates |

---
### Model Assumptions
This framework assumes:
- Renting and owning provide equivalent housing utility
- Market participants price homes using expected appreciation
- Ownership costs can be approximated by fixed depreciation assumptions
- Mortgage rates proxy financing cost.

### Limitations
- Ex-post appreciation used as validation rather than expectation
- Non-financial ownership benefits excluded
- Local supply constraints not explicitly modeled

---
### Practical Implications
These findings are relevant for:

**Homebuyers:**
Assess whether purchase decisions rely heavily on future appreciation assumptions

**Investors:**
Identify markets with weaker rent-supported pricing

**Policy analysts:**
Identify affordability stress and evaluate where pricing dynamics may be diverging from rent-supported benchmarks

---

### Analytical Takeaways
This project reinforced:

- The importance of validating model assumptions
- The sensitivity of valuation outputs to appreciation estimates
- The limits of equilibrium-based pricing models in fast-growth housing markets


---
### Analytical Roadmap
1. **Framing the Valuation Problem:**
2. **Building the Market Dataset:**
3. **Establishing Historical Context:**
4. **Case Study: Dallas 2020:**
5. **Scaling Across Texas Markets:**
6. **Identifying Pricing Regimes:**
7. **Tracking Market Disequilibrium Over Time:**
8. **Interpreting Market Signals:**
9. **Stress Testing the Model:**
10. **Final Takeaways:**


---
## Step 1 — Setup: Mount Drive & Import Libraries

We're running in Google Colab, so the first thing we do is mount Google Drive so the notebook can access CSV files stored there. Then we import the two libraries we'll use throughout:
- **pandas** for loading and manipulating tabular data
- **matplotlib** for plotting charts

In [ ]:
# Mount Google Drive so we can read files from it
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd     # For loading CSVs and data manipulation
import matplotlib.pyplot as plt  # For plotting charts

CITIES = ['Dallas, TX', 'Houston, TX', 'Austin, TX', 'San Antonio, TX']

---
## Step 2 — Load the Datasets

We load three datasets, each from a CSV file stored in Google Drive.

### 2a. ZHVI — Zillow Home Value Index
Each row is a metro area. Columns after the metadata fields are monthly dates (e.g. `2000-01-31`), and the values are the **median home price** for that metro in that month.

> The dataset spans January 2000 through early 2026 — 320 columns total.

In [ ]:
# Load Zillow Home Value Index (median home prices by metro area, monthly)
zhvi_df = pd.read_csv('/content/drive/MyDrive/Datasets/Zillow_Research/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv')

print('ZHVI DataFrame loaded:')
display(zhvi_df.head())
print(f'Shape: {zhvi_df.shape}')    # Rows = metro areas, Cols = metadata + monthly dates

### 2b. ZORI — Zillow Observed Rent Index
Same structure as ZHVI, but values are **median monthly rents** (not home prices). We'll multiply by 12 to annualize rent when needed.

In [ ]:
# Load Zillow Observed Rent Index (median monthly rents by metro area)
zori_df = pd.read_csv('/content/drive/MyDrive/Datasets/Zillow_Research/Metro_zori_uc_sfrcondomfr_sm_month.csv')

print('ZORI DataFrame loaded:')
display(zori_df.head())
print(f'Shape: {zori_df.shape}')  # Same thing

### 2c. FRED — 30-Year Mortgage Rates
Weekly observations of the national average 30-year fixed mortgage rate, sourced from the Federal Reserve Economic Data (FRED). We'll average these to get a single annual rate for each year.

In [ ]:
# Load FRED 30-year mortgage rate data (weekly observations)
fred_df = pd.read_csv('/content/drive/MyDrive/Datasets/Zillow_Research/MORTGAGE30US.csv')

print('FRED Mortgage Rate DataFrame loaded:')
display(fred_df.head())

### 2d. Verifying Analytical Inputs

Before proceeding, we run a quick sanity check on all three datasets —
confirming row counts, date ranges, null values, and that all four cities
we plan to analyze are actually present in the data.

In [ ]:
# --- Data validation: confirm each dataset loaded as expected ---

print("DATASET VALIDATION SUMMARY")
print('=' * 55)
print('ZHVI')
print(f'  Rows (metro areas): {zhvi_df.shape[0]}')
date_cols_zhvi = [c for c in zhvi_df.columns if c[0].isdigit()]
print(f'  Date range: {date_cols_zhvi[0]} → {date_cols_zhvi[-1]}')
print(f'  TX MSA rows: {len(zhvi_df[(zhvi_df["StateName"]=="TX") & (zhvi_df["RegionType"]=="msa")])}')

print()
print('ZORI')
print(f'  Rows (metro areas): {zori_df.shape[0]}')
date_cols_zori = [c for c in zori_df.columns if c[0].isdigit()]
print(f'  Date range: {date_cols_zori[0]} → {date_cols_zori[-1]}')
print(f'  TX MSA rows: {len(zori_df[(zori_df["StateName"]=="TX") & (zori_df["RegionType"]=="msa")])}')

print()
print('FRED')
print(f'  Rows (weekly readings): {fred_df.shape[0]}')
print(f'  Date range: {fred_df["observation_date"].min()} → {fred_df["observation_date"].max()}')
print(f'  Null mortgage rate values: {fred_df["MORTGAGE30US"].isna().sum()}')

print()
print('City coverage check (ZHVI):')
for city in CITIES:
    found = city in zhvi_df['RegionName'].values
    print(f'  {city}: {"FOUND" if found else "NOT FOUND"}')

print()
print('City coverage check (ZORI):')
for city in CITIES:
    found = city in zori_df['RegionName'].values
    print(f'  {city}: {"FOUND" if found else "NOT FOUND"}')

print()
print("Check for duplicate city rows for both ZHVI and ZORI:")
# Check for duplicate city names in ZHVI and ZORI
zhvi_dupes = zhvi_df[zhvi_df['RegionName'].isin(CITIES)].groupby('RegionName').size()
zori_dupes = zori_df[zori_df['RegionName'].isin(CITIES)].groupby('RegionName').size()

print('ZHVI city row counts (expected 1 each):')
print(zhvi_dupes)
print()
print('ZORI city row counts (expected 1 each):')
print(zori_dupes)

## Step 3 — Explore: Texas Home Prices (2010–2019)

Before building the model, we plot median home prices for our four cities across
the full analysis window. This gives us a visual baseline for the price levels
and growth rates each city experienced — context that will matter when
interpreting the regime classifications later.

A few things to note going in: Austin starts the period as the most expensive
of the four metros and pulls away sharply after 2012, consistent with its early
and persistent entry into the speculation-dominated regime. Dallas and Houston
track closely through most of the period. San Antonio remains the most affordable
throughout.

In [ ]:
city_zhvi = zhvi_df[
    (zhvi_df['RegionName'].isin(CITIES)) &
    (zhvi_df['RegionType'] == 'msa')
]

# Pull only the mid-year columns (June 30) from 2010–2019 for a clean annual view
date_cols = [col for col in city_zhvi.columns if col.endswith('06-30')
             and 2010 <= int(col[:4]) <= 2019]

plt.figure(figsize=(12, 6))

for _, row in city_zhvi.iterrows():
    prices = [row[col] for col in date_cols]
    years  = [int(col[:4]) for col in date_cols]
    plt.plot(years, prices, marker='o', label=row['RegionName'])

plt.xlabel('Year')
plt.ylabel('Median Home Price ($)')
plt.title('Texas Metro Home Prices (2010–2019)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## Step 4 — Single-City Deep Dive: Dallas, 2020

Now for the core analysis. We'll work through the **rent-vs-own equilibrium model** step by step, using Dallas in 2020 as our example.

### The Formula

For a buyer and renter to be financially indifferent between owning and renting, the following must hold:

$$g = r + d - \frac{R}{P}$$

| Variable | Meaning |
|---|---|
| **g** | Implied annual home price growth (what the market is pricing in) |
| **r** | 30-year mortgage rate (cost of borrowing) |
| **d** | Depreciation + maintenance (we use 3%) |
| **R** | Annual rent for an equivalent home |
| **P** | Current home price |

**Intuition:** The more you pay for a home relative to what it rents for, the higher the future appreciation needs to be to justify that price. Low rent-to-price ratio → market is betting on big gains.

> **Why Dallas 2020?** We use Dallas in 2020 as the worked example for this
> deep dive because it's one of the clearest cases where the model produces
> a fully interpretable result — all three inputs (P, R, and r) are available,
> and the implied vs. actual growth comparison tells a clean story. Step 5 will
> show that this is the exception rather than the rule: for most Texas cities
> and years in 2010–2019, the model either lacks rent data or breaks down
> entirely due to speculation-dominated growth.

---

### Step 4a — Pull Home Price (P) and Annual Rent (R)

In [ ]:
# --- Configuration: set the city and year we're analyzing ---
CITY = 'Dallas, TX'
YEAR = 2020

# Depreciation + maintainence constant (standard assumption: ~3% of home value per year)
d = 0.03

# We use mid-year (June 30) as our snapshot date for a stable, non-seasonal reading
date_snapshot = f'{YEAR}-06-30'


In [ ]:
# --- Pull Median Home Price (P) from ZHVI ---
# Filter for our city; ensure we're looking at metro-level data (not county, etc.)
city_zhvi = zhvi_df[
    (zhvi_df['RegionName'] == CITY) &
    (zhvi_df['RegionType'] == 'msa')
]

if date_snapshot in city_zhvi.columns and not city_zhvi.empty:
  P = city_zhvi[date_snapshot].iloc[0]  # .iloc[0] gets the single matching row's value
  print(f'Median Home Price (P) for {CITY} in {date_snapshot}: ${P:,.2f}')
else:
  P = None
  print(f'No ZHVI data found for {CITY} on {date_snapshot}.')

In [ ]:
# --- Pull Median Monthly Rent from ZORI and annualize it ---
city_zori = zori_df[
    (zori_df['RegionName'] == CITY) &
    (zori_df['RegionType'] == 'msa')
]

if date_snapshot in city_zori.columns and not city_zori.empty:
  R_monthly = city_zori[date_snapshot].iloc[0]
  R = R_monthly * 12  # Convert monthly rent to annual so units match home price
  print(f'Median Annual Rent (R) for {CITY} in {date_snapshot}: ${R:,.2f}')
else:
  R = None
  print(f'No ZORI data found for {CITY} on {date_snapshot}.')


### Step 4b — Pull Mortgage Rate (r) from FRED

FRED gives us weekly mortgage rate readings. We average all the weeks in our target year to get a single representative annual rate.

In [ ]:
# Convert the date column to proper datetime objects so we can filter by year
fred_df['observation_date'] = pd.to_datetime(fred_df['observation_date'])

# Keep only the rows from our target year
fred_year = fred_df[fred_df['observation_date'].dt.year == YEAR]

if not fred_year.empty:
  # Average all weekly readings for the year, then convert from % to decimal
  # e.g., 3.11% -> 0.0311
  r = fred_year['MORTGAGE30US'].mean() / 100
  print(f'Average 30-year mortgage rate (r) for {YEAR}: ({r*100:.2f}%)')
else:
  r = None
  print(f'No FRED mortgage data found for {YEAR}.')

### Step 4c — Solve for Implied Growth Rate (g)

With P, R, r, and d in hand, we can now solve the equilibrium formula:

$$g = r + d - \frac{R}{P}$$

The result tells us: **"At 2020 Dallas prices, what annual appreciation does the market need to price in for buying to equal renting?"**

- If `g` is high → the market is betting on strong future gains (potentially overvalued)
- If `g` is low → the market expects modest gains (potentially undervalued or fairly priced)

In [ ]:
# Only calculate if all three inputs are available
if P is not None and R is not None and r is not None and P != 0:

  # Apply the equilibrim formula: g = r + d - (R / P)
  g = r + d - (R / P)

  print(f'Implied growth rate (g) for {CITY} in {YEAR}: {g:.4f} ({g*100:.2f}%)')
  print()
  print('Interpretation: this is the annual home price appreciation the market')
  print('must be assuming for buying to make as much financial sense as renting.')
else:
  g = None
  print('Cannot calculate implied growth rate - missing P, R, or r.')

### Step 4d — Compare to Actual 5-Year Growth

Now we find out what *actually* happened: did Dallas homes appreciate at the rate the 2020 market implied, or did reality diverge?

We calculate the **Compound Annual Growth Rate (CAGR)** from 2020 to 2025:

$$g_{actual} = \left(\frac{P_{final}}{P_{initial}}\right)^{\frac{1}{n}} - 1$$

where `n = 5` years.

In [ ]:
P_initial = P           # Starting home price (from the snapshot year)
YEAR_FINAL = YEAR + 5   # We look 5 years forward
date_final = f'{YEAR_FINAL}-06-30'

# Pull the home price 5 years later (same city, same mid-year snapshot)
city_zhvi_final = zhvi_df[
    (zhvi_df['RegionName'] == CITY) &
    (zhvi_df['RegionType'] == 'msa')
]

if date_final in city_zhvi_final.columns and not city_zhvi_final.empty:
  P_final = city_zhvi_final[date_final].iloc[0]
  print(f'Home price in {date_final}: ${P_final:,.2f}')

  if P_initial is not None and P_initial != 0:
    n_years = YEAR_FINAL - YEAR

    # What constant yearly growth rate would turn P_initial into P_final over 5 years?
    # CAGR formula: ((end / start) ^ (1/n)) - 1
    g_actual = ((P_final / P_initial) ** (1 / n_years)) - 1
    print(f'Actual annualized growth (g_actual) from {YEAR} to {YEAR_FINAL}: {g_actual:.4f} ({g_actual*100:.2f}%)')

    # --- Verdict: did the market under- or over-estimate future gains? ---
    if g is not None:
      print()
      print(f'  Implied growth (g):     {g*100:.2f}%')
      print(f'  Actual growth (g_actual): {g_actual*100:.2f}%')
      print()

      if abs(g - g_actual) < 0.01:    # Within 1 percentage point = "close"
        print('The market was roughly accurate in its growth expectations.')
      elif g > g_actual:
        print('The market was OPTIMISTIC - homes were priced for more growth than actually occurred (potentially overvalued).')
      else:
        print('The market was PESSIMISTIC - homes appreciated much faster than prices implied (potentially undervalued).')
else:
  print(f'No ZHVI data found for {CITY} on {date_final}.')

### Step 4e — Visualize: Implied vs. Actual Growth

A simple bar chart to make the comparison between implied and actual growth visually obvious.

In [ ]:
# Only plot if both values were successfully calculated
if g is not None and g_actual is not None:
  labels = ['Implied Growth Rate (g)', 'Actual 5-Year Growth Rate (g_actual)']
  values = [g * 100, g_actual * 100]    # Convert decimals to percentages for display
  colors = ['skyblue', 'lightcoral']


plt.figure(figsize=(8, 5))
bars = plt.bar(labels, values, color=colors, width=0.4)

# Add value labels on top of each bar
for bar, val in zip(bars, values):
  plt.text(
      bar.get_x() + bar.get_width() / 2,   # Center the label horizontally
      bar.get_height() + 0.1,              # Place it just above the bar
      f'{val:.2f}%',
      ha='center', va='bottom', fontsize=11
  )



plt.ylabel('Annualized Growth Rate (%)')
plt.title(f'Implied vs. Actual Home Price Growth - {CITY}, {YEAR}')
plt.ylim(0, max(values) * 1.2)    # Add headroom above the tallest bar
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

---
## Step 5 — Multi-City, Multi-Year Analysis (2010–2019)

We now repeat the same analysis across **four Texas cities** and **ten years** (2010–2019), building a comprehensive table.

For each city/year combination we collect:
- **P** — median home price (ZHVI, mid-year snapshot)
- **R** — median annual rent (ZORI × 12)
- **r** — average 30-year mortgage rate for that year (FRED)
- **g_actual** — actual annualized 5-year home price growth (CAGR)

> **Note on missing data:** Dallas's ZORI data doesn't start until 2014, so `R` (and downstream calculations) will be `None` for Dallas 2010–2013.

In [ ]:
# --- Configuration ---
YEARS = list(range(2010, 2020))   # A list from 2010 through 2019
d = 0.03                          # Depreciation constant (same as before)
n_years = 5                       # The years in which we look forward

# Parse FRED dates once upfront so we don't repeat it in every loop iteration
fred_df['observation_date'] = pd.to_datetime(fred_df['observation_date'])   # converting to datetime

results_list = []    # We'll collect one dict per city/year, then convert to a DataFrame

for CITY in CITIES:
  # Pre-filter to this city's rows once, outside the year loop (faster than filtering inside)
  city_zhvi_rows = zhvi_df[(zhvi_df['RegionName'] == CITY) & (zhvi_df['RegionType'] == 'msa')]
  city_zori_rows = zori_df[(zori_df['RegionName'] == CITY) & (zori_df['RegionType'] == 'msa')]

  for YEAR in YEARS:
    date_snapshot = f'{YEAR}-06-30'
    date_final = f'{YEAR + n_years}-06-30'

    # --- a. Median Home Price (P) ---
    if date_snapshot in city_zhvi_rows.columns and not city_zhvi_rows.empty:
      P = city_zhvi_rows[date_snapshot].iloc[0]
    else:
      P = None

    # --- b. Median Annual Rent (R) ---
    # ZORI is monthly; multiply by 12 to annualize
    if date_snapshot in city_zori_rows.columns and not city_zori_rows.empty:
      R_monthly = city_zori_rows[date_snapshot].iloc[0]
      R = R_monthly * 12 if pd.notna(R_monthly) else None
    else:
      R = None
    # --- c. Average Mortgage Rate (r) ---
    fred_year = fred_df[fred_df['observation_date'].dt.year == YEAR]
    r = fred_year['MORTGAGE30US'].mean() / 100 if not fred_year.empty else None

    # --- d. Actual 5-Year Annualized Growth (g_actual) ---
    if (date_final in city_zhvi_rows.columns and not city_zhvi_rows.empty and P is not None and P != 0):
      P_final = city_zhvi_rows[date_final].iloc[0]
      g_actual = ((P_final / P) ** (1 / n_years)) - 1  # CAGR over 5 years
    else:
      g_actual = None

    # --- e. Store results for this city/year ---
    results_list.append({
        'City': CITY,
        'Year': YEAR,
        'P': P,               # Home price
        'R': R,               # Annual rent
        'r': r,               # Mortgage rate (decimal)
        'g_actual': g_actual  # Actual 5-yr annualized growth
    })


# Convert list of dicts into a clean DataFrame
df_analysis = pd.DataFrame(results_list)

print('Multi-city analysis results:')
display(df_analysis)

In [ ]:
# --- Cleaning up for Presentation ---
presentation_df = df_analysis.copy().rename(columns={
    'P': 'Median Home Price',
    'R': 'Annual Rent',
    'r': 'Mortgage Rate',
    'g_actual': 'Actual 5-Yr Growth'
})

presentation_df['Median Home Price'] = presentation_df['Median Home Price'].map(lambda x: f'${x:,.0f}' if pd.notna(x) else 'N/A')
presentation_df['Annual Rent'] = presentation_df['Annual Rent'].map(lambda x: f'${x:,.0f}' if pd.notna(x) else 'N/A')
presentation_df['Mortgage Rate'] = presentation_df['Mortgage Rate'].map(lambda x: f'{x*100:.2f}%' if pd.notna(x) else 'N/A')
presentation_df['Actual 5-Yr Growth'] = presentation_df['Actual 5-Yr Growth'].map(lambda x: f'{x*100:.2f}%' if pd.notna(x) else 'N/A')

display(presentation_df)

In [ ]:
city_summary = df_analysis.groupby('City')['g_actual'].mean().reset_index()
city_summary['g_actual'] = city_summary['g_actual'].map(lambda x: f'{x*100:.2f}%')

if not city_summary.empty:
  # Labels for the x-axis (city names)
  plot_labels = city_summary['City']

  # Values for the y-axis (average actual growth rates)
  # Convert 'g_actual' from string ('8.18%') to float (8.18)
  plot_values = city_summary['g_actual'].str.replace('%', '').astype(float)

  # sort the rates in descending order
  plot_values = plot_values.sort_values(ascending=False)
  plot_labels = plot_labels[plot_values.index]

  plt.figure(figsize=(10, 6))
  bars = plt.bar(plot_labels, plot_values, color='skyblue', width=0.4)

  # Add value labels on top of each bar
  for bar, val in zip(bars, plot_values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,   # Center the label horizontally
        bar.get_height() + 0.1,              # Place it just above the bar
        f'{val:.2f}%',                       # Display with '%'
        ha='center', va='bottom', fontsize=11
    )

  plt.ylabel('Average Annualized Growth (%)')
  plt.title('Average 5-Year Home Price Growth by Texas Metro')
  plt.ylim(0, max(plot_values) * 1.2)    # Add headroom above the tallest bar
  plt.grid(axis='y', linestyle='--', alpha=0.6)
  plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability
  plt.tight_layout()
  plt.show()
else:
  print("No data available in city_summary to plot.")

## Step 6 — Calculate the R/u Ratio

### What is Implied Rent (u)?
Implied rent (`u`) is the annual rent that would make a buyer *exactly indifferent* between renting and buying — given the actual future appreciation that occurred.

$$u = P \times (r + d - g_{actual})$$

This is the formula `g = r + d - R/P` rearranged: instead of solving for the implied growth rate, we're solving for the implied rent.

### The R/u Ratio

$$\frac{R}{u} = \frac{\text{Actual annual rent}}{\text{Implied annual rent}}$$

| R/u | Interpretation |
|---|---|
| ≈ 1 | Balanced market — buying and renting are roughly equivalent |
| < 1 | Owning is expensive relative to renting (market pricing in high appreciation) — potential overvaluation |
| > 1 | Owning is cheap relative to renting (market pricing in low appreciation) — potential undervaluation |

### Why Some Rows Are Missing

Two distinct reasons R/u may be uncalculable for a given city/year:

| Cause | Example | What it means |
|---|---|---|
| **R is NaN** | Dallas 2010–2013 | Zillow didn't collect rent data — no fix available |
| **g_actual > r + d** | Dallas 2017, Austin 2027 | Appreciation was so extreme the model breaks down — these rows are classified as **speculation-dominated** |

Rather than silently dropping these rows, the code assigns each a `regime` label so they remain visible in the table and are flagged with shading on the chart.

In [ ]:
# --- Helper: classify each row's market regime ---
def classify_regime(row, d=0.03):
    """
    Returns a regime label for each city/year based on the relationship
    between actual growth, mortgage rate, and the R/u ratio.

    Priority order:
      1. No data        — missing g_actual or r (can't assess anything)
      2. Speculation    — g_actual so high the model breaks down (u goes negative)
      3. Overvalued     — R/u < 1 (buying expensive relative to renting)
      4. Undervalued    — R/u > 1 (buying cheap relative to renting)
      5. Balanced       — R/u ≈ 1
    """
    if pd.isna(row['g_actual']) or pd.isna(row['r']):
      return 'No data'
    elif row['g_actual'] > (row['r'] + d):
      return 'Speculation-dominated'
    elif pd.isna(row['R/u']):         # R missing but not speculation
      return 'No data'
    elif row['R/u'] < 1:
      return 'Overvalued'
    elif row['R/u'] > 1:
      return 'Undervalued'
    elif abs(row['R/u'] - 1) < 0.05:
      return 'Balanced'


In [ ]:
# --- Calculate implied rent (u) ---
def calc_implied_rent(row, d=0.03):
  """
    u = P * (r + d - g_actual)
    Returns None if any input is missing or if g_actual exceeds r + d
    (speculation-dominated regime - u would be zero or negative).
  """
  if any(pd.isna(x) for x in [row['P'], row['r'], row['g_actual']]):
    return None
  u = row['P'] * (row['r'] + d - row['g_actual'])
  return u if u > 0 else None

In [ ]:
# --- Calculate R/u ratio ---
def calc_ru_ratio(row):
  """
    R/u - actual rent divided by implied rent.
    Returns None if either value is missing.
  """
  if pd.isna(row['R']) or pd.isna(row['u']):
    return None
  return row['R'] / row['u']

In [ ]:
# Apply both calculations column-by-column
df_analysis['u'] = df_analysis.apply(calc_implied_rent, axis=1)
df_analysis['R/u'] = df_analysis.apply(calc_ru_ratio, axis=1)

# Add an extra column for classification
df_analysis['speculation_regime'] = df_analysis['g_actual'] > (df_analysis['r'] + 0.03)
df_analysis['regime'] = df_analysis.apply(classify_regime, axis=1)

print('Analysis table with regime labels:')
display(df_analysis[['City', 'Year', 'P', 'R', 'r', 'g_actual', 'u', 'R/u', 'regime']])

In [ ]:
# --- For Presentation Purposes ---
presentation_df = df_analysis.copy().rename(columns={
    'P': 'Median Home Price',
    'R': 'Annual Rent',
    'r': 'Mortgage Rate',
    'g_actual': 'Actual Growth',
    'u': 'Implied Rent',
    'R/u': 'Rent-to-Rent Ratio'
})

presentation_df['Median Home Price'] = presentation_df['Median Home Price'].map(lambda x: f'${x:,.0f}' if pd.notna(x) else 'N/A')
presentation_df['Annual Rent'] = presentation_df['Annual Rent'].map(lambda x: f'${x:,.0f}' if pd.notna(x) else 'N/A')
presentation_df['Mortgage Rate'] = presentation_df['Mortgage Rate'].map(lambda x: f'{x*100:.2f}%' if pd.notna(x) else 'N/A')
presentation_df['Actual Growth'] = presentation_df['Actual Growth'].map(lambda x: f'{x*100:.2f}%' if pd.notna(x) else 'N/A')
presentation_df['Implied Rent'] = presentation_df['Implied Rent'].map(lambda x: f'${x:,.0f}' if pd.notna(x) else 'N/A')
presentation_df['Rent-to-Rent Ratio'] = presentation_df['Rent-to-Rent Ratio'].map(lambda x: f'{x:.2f}' if pd.notna(x) else 'N/A')

presentation_df.sort_values(['City', 'Year'])

display(
    presentation_df[
        [
            'City',
            'Year',
            'Median Home Price',
            'Annual Rent',
            'Mortgage Rate',
            'Actual Growth',
            'Implied Rent',
            'Rent-to-Rent Ratio',
            'regime'
        ]
    ]
)

## Step 7 — Visualize the R/u Ratio Over Time

Now we plot the R/u ratio for all four cities from 2010 to 2019 on the same chart.
The **red dashed line at y=1** marks the equilibrium point — markets above it were
undervalued, below it overvalued, relative to actual subsequent appreciation.
**Orange shaded columns** mark years where the model breaks down due to
speculation-dominated growth (`g_actual > r + d`); **grey columns** mark years with
no rent data. Gaps in a city's line during shaded years are expected, not errors.

> **Note:** Only 5 of 40 city/year combinations produce a valid R/u ratio. The
> dominant reason is that Texas home prices appreciated so rapidly across this
> period that `g_actual > r + d` for most cities and years — placing them in the
> speculation-dominated regime where the rent-vs-own framework stops being a
> useful lens. This itself is a finding: Texas metros were not priced on rental
> fundamentals for most of 2010–2019.

In [ ]:
# --- Plot: R/u over time with regime shading ---

# Color map for the regime shading bands
REGIME_COLORS = {
    'Speculation-dominated': 'orange',
    'No data': 'grey',
}

# Cap R/u at a reasonable ceiling to prevent outliers from distorting the chart
RU_CAP = 20
df_plot = df_analysis.dropna(subset=['R/u']).copy()
df_plot['R/u'] = df_plot['R/u'].clip(upper=RU_CAP)

# Warn about any rows that were capped so we know which ones were affected
capped = df_analysis[df_analysis['R/u'] > RU_CAP][['City', 'Year', 'R/u']]
if not capped.empty:
    print('The following R/u values were capped for display:')
    print(capped)

plt.figure(figsize=(12, 6))

# Plot one line per city
for city in CITIES:
    city_data = df_plot[df_plot['City'] == city]
    plt.plot(
        city_data['Year'],
        city_data['R/u'],
        marker='o',
        label=city
    )

# Shade once per year using the most severe regime present that year
# We iterate over ALL rows (not just df_plot) so years dropped from the
# line due to negative u are still shaded correctly
shaded_labels = set()
for year, year_group in df_analysis.groupby('Year'):
    regimes_this_year = year_group['regime'].unique()

    for regime in ['Speculation-dominated', 'No data']:  # Priority order: most severe first
        if regime in regimes_this_year:
            color = REGIME_COLORS[regime]
            label = regime if regime not in shaded_labels else None
            plt.axvspan(year - 0.4, year + 0.4, color=color, alpha=0.15, label=label)
            shaded_labels.add(regime)
            break  # Use only the most severe regime label per year; don't stack bands

# Equilibrium reference line
plt.axhline(y=1, color='red', linestyle='--', linewidth=1.5, label='Equilibrium (R/u = 1)')

# Green/red background zones pinned to axis bounds for consistent visibility
plt.axhspan(0, 1, alpha=0.05, color='red')        # Below equilibrium = overvalued
plt.axhspan(1, RU_CAP, alpha=0.05, color='green') # Above equilibrium = undervalued

plt.xlabel('Year')
plt.ylabel('R/u Ratio')
plt.ylim(0, RU_CAP)
plt.title('R/u Ratio Over Time - Texas Metro Areas (2010-2019)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

> **Note on Dallas:** Dallas produces no valid R/u values for the entire
> 2010–2019 period. Years 2010–2014 lack ZORI rent data, and years 2015–2019
> fall into the speculation-dominated regime (`g_actual > r + d`). Dallas
> home prices appreciated so consistently and strongly that the rent-vs-own
> framework could not establish a valid equilibrium price for any year in
> this window.

In [ ]:
# --- Print a quick summary of how many city/years fall in each regime ---
print('\nRegime breakdown:')
print(df_analysis.groupby('regime')[['City']].count().rename(columns={'City': 'Count'}))

## Step 8 — Summary & Interpretation

### What the R/u Ratio Tells Us

**R/u ≈ 1 (Balanced market)**
The actual rent and the implied rent are in sync. The market is efficiently priced — buying vs. renting is roughly a wash at current prices, given actual future appreciation.

**R/u < 1 (Potentially overvalued)**
Actual rents are *lower* than what would be needed to justify home prices. Buyers are implicitly betting on strong future appreciation. If that appreciation doesn't materialize, prices are unsustainably high.

**R/u > 1 (Potentially undervalued)**
Actual rents are *higher* than implied rent — meaning rents alone nearly or fully justify the home price. Buyers don't need to rely on price appreciation to get a fair deal. This is often a buy signal.

---

### Key Findings

- **Dallas 2020 (single-city deep dive):** The market implied only **0.23% annual growth**, but homes actually appreciated at **6.84%/year** over the next 5 years. Dallas pricing appears increasingly difficult to reconcile with rent-based valuation alone, suggesting market expectations of continued appreciation played a dominant role.
- **Dallas 2010–2019:** Dallas produces no valid R/u values for the entire analysis window. Years 2010–2014 lack ZORI rent data, and years 2015–2019 fall into the speculation-dominated regime (`g_actual > r + d`). Dallas home prices appreciated so consistently and strongly that the rent-vs-own framework could not establish a valid equilibrium for any year in this period.
- **Speculation dominates the dataset:** Of 40 total city/year combinations, 24 are speculation-dominated and 11 have no rent data — leaving only 5 rows with a valid R/u ratio. This is itself a finding: Texas metros were not priced on rental fundamentals for most of 2010–2019.
- **The 5 valid observations (Houston 2015–2016, Austin 2015, San Antonio 2015, 2019)** all show R/u well above 1 (ranging ~3–7), indicating homes were meaningfully undervalued relative to rents in those years — consistent with the strong appreciation that followed.

---

### Conclusion

Texas housing markets during 2010–2019 were largely **not priced on rental fundamentals**. Of the 40 city/year combinations analyzed, 24 fell into a speculation-dominated regime where home price appreciation outpaced the combined cost of borrowing and depreciation (`g_actual > r + d`), rendering the rent-vs-own framework inapplicable. A further 11 lacked sufficient rent data to assess at all — leaving only 5 valid observations across four cities and ten years.

The 5 years where the model could be applied — Houston 2015–2016, Austin 2015, and San Antonio 2015 and 2019 — all produced R/u ratios well above 1 (ranging ~3–7), indicating that homes were **meaningfully undervalued relative to rents** in those windows. Buyers in those years did not need to rely on future appreciation to justify their purchase; rents alone nearly or fully supported the price paid. The strong appreciation that followed in all four metros is consistent with this diagnosis.

Dallas, the largest metro in the sample, produced no valid R/u values for the entire period — a finding in itself. Its uninterrupted presence in the speculation-dominated regime from 2015 onward reflects a market where prices were driven almost entirely by appreciation expectations rather than rental income fundamentals.

Taken together, the evidence suggests that Texas metros spent most of the 2010–2019 decade in a regime where traditional rent-vs-own valuation breaks down. This does not mean prices were irrational — strong in-migration, job growth, and constrained housing supply all provided real economic foundations for appreciation. But it does mean that buyers in this period were implicitly making a bet on continued price growth, whether they knew it or not.

---



## Step 9 — Sensitivity Analysis: How Much Does `d` Matter?

Throughout this analysis we assumed a depreciation and maintenance constant of
`d = 0.03` (3%). This is a standard assumption, but it directly affects which
city/years are classified as speculation-dominated vs. undervalued — because the
regime boundary is `g_actual > r + d`.

A higher `d` raises that boundary, meaning fewer years get classified as
speculation-dominated. A lower `d` lowers it, meaning more years cross into
the speculation regime. The table below shows how sensitive our regime counts
are to this assumption.

In [ ]:
# Test how regime classifications change across different values of d
D_VALUES = [0.02, 0.03, 0.04]

sensitivity_rows = []

for d_test in D_VALUES:
    for _, row in df_analysis.iterrows():
        if pd.isna(row['g_actual']) or pd.isna(row['r']):
            regime = 'No data'
        elif row['g_actual'] > (row['r'] + d_test):
            regime = 'Speculation-dominated'
        else:
            regime = 'Undervalued/Balanced'

        sensitivity_rows.append({
            'd': f'{d_test:.0%}',
            'City': row['City'],
            'Year': row['Year'],
            'regime': regime
        })

df_sensitivity = pd.DataFrame(sensitivity_rows)

# Pivot to a summary table: how many rows fall in each regime per value of d
summary = df_sensitivity.groupby(['d', 'regime']).size().unstack(fill_value=0)
print('Regime counts by depreciation assumption:')
display(summary)

### What the Sensitivity Analysis Shows

The classification of city/years as speculation-dominated is **moderately sensitive**
to the choice of `d`. At our baseline of 3%, 24 of 29 data-sufficient rows are
speculation-dominated. Raising `d` to 4% — a reasonable upper bound for older
housing stock — drops that to 14, recovering 10 rows as undervalued. Lowering `d`
to 2% pushes 4 more rows into the speculation regime, leaving only 12 undervalued.

The core conclusion holds across all three values: Texas metros spent the majority
of 2010–2019 in a speculation-dominated regime. But the margin is sensitive enough
that `d` is worth flagging as a key assumption rather than a settled constant.

### Next Steps

- **Acquire Dallas rent data pre-2014** (or impute it) to complete the full 2010–2019 picture for Dallas.
- **Extend analysis post-2019** to capture the COVID-era housing surge (2020–2022) and subsequent correction — a period where speculation-dominated regimes would likely be even more prevalent.
- **Compare Texas metros to the national average** using the `United States` row in ZHVI to contextualize whether TX was relatively cheap or expensive vs. the country.